## Preprocesamiento y Modelado

Una vez inspeccionado el dataset en `customer_churn_eda.ipynb` definimos una una estrategia de preprocesamiento iterativo (de menos a más) para el encontrar

In [ ]:
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.preprocessing import OneHotEncoder,RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, f1_score, make_scorer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.discriminant_analysis import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB


### Configuración de constantes, rutas y variables 

En esta sección definimos constantes, rutas de archivos y atributos del dataset

In [24]:
# Rutas de los archivos de datos
TRAIN_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/train.csv"
TEST_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Cargamos los datos
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
# Definir variables objetivo
TARGET = 'Exited'
# Variables numéricas: Incluyo las continuas y las binarias numéricas (HasCrCard, IsActiveMember)
NUM_FEATURES = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
# Variables categóricas: Las de texto con pocas categorías
CAT_FEATURES = ['Geography', 'Gender']
# Variables a eliminar inicialmente (IDs y apellido)
DROP_FEATURES = ['CustomerId', 'Surname']
SURNAME_COL = 'Surname'
RANDOM_STATE = 100            # Semilla para reproducibilidad

### 1. Preparación de Datos

Separamos variables independientes y dependientes en X_train e y_train por convención.

In [25]:
# X_train = variables independientes
# y_train = variable dependiente u objetivo
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

---------------------------------------

### Funciones auxiliar
#### Construir preprocesadores

In [26]:
def make_preprocessor(X: pd.DataFrame, version: str,numerical_cols:[], categorical_cols:[]):
    """Genera un ColumnTransformer con pipelines de preprocesamiento
    para variables numéricas y categóricas según la versión indicada.

    Args:
        X (pd.DataFrame): _input data frame_
        version (str): Versión del preprocesamiento en formato 'N#_C#'
        e.g. 'N3_C1' donde N# indica la versión numérica y C# la categórica
        numerical_cols (_type_): columnas numéricas para el preprocesamiento
        categorical_cols (_type_): columnas categóricas para el preprocesamiento

    Raises:
        ValueError: _unknown numeric version_
        ValueError: _unknown categorical version_

    Returns:
        _type_: ColumnTransformer con pipelines de preprocesamiento
    """
    num_version, categorical_version = version.split("_")  # e.g. 'N3', 'C1'

    # --- Numerical pipeline ---
    numerical_transformers = []

    if num_version == "N1":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con mediana
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N2":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con mediana + indicador de faltantes
        # add_indicator=True añade una columna booleana por cada numérica
        # que indica si el valor estaba faltante
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N3":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con mediana + indicador de faltantes + escalado estandar
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N4":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con mediana + escalado estandar
        # Escalado robusto a outliers con RobustScaler
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N5":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        num_pipe = Pipeline(steps=[
            ('scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="uniform"))
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N6":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        # Escalado robusto a outliers con RobustScaler
        num_pipe = Pipeline(steps=[
            ('pre_scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="uniform")),
            ("scaler", RobustScaler()),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N7":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con KNN + escalado RobustScaler
        # KNN requiere escalar primero para calcular distancias bien.
        num_pipe = Pipeline(steps=[
            ('scaler', RobustScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="uniform")),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    
    elif num_version == "N8":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        num_pipe = Pipeline(steps=[
            ('scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=9, weights="uniform"))
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N9":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        # add_indicator=True añade una columna booleana por cada numérica
        # que indica si el valor estaba faltante
        # añadir indicador puede ayudar si hubiera correlación entre faltantes y target
        num_pipe = Pipeline(steps=[
            ('pre_scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="uniform",add_indicator=True)),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N10":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        # add_indicator=True añade una columna booleana por cada numérica
        # que indica si el valor estaba faltante
        # vecinos más cercanos pesan más puede que la imputación sea más fina (si hay grupos claros)
        num_pipe = Pipeline(steps=[
            ('pre_scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="distance",add_indicator=True)),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    
    else:
        raise ValueError(f"Unknown numeric version: {num_version}")

    # --- Categorical pipeline (si aplica) ---
    categorical_transformers = []
    if categorical_version == "C0": 
        # No aplica pipeline en categoricas
        pass
    elif categorical_version == "C1":
        # Categóricas normales: imputar + onehot
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        categorical_transformers.append(("cat", categorical_pipe, categorical_cols))

    else:
        raise ValueError(f"Unknown categorical version: {categorical_version}")

    # --- Build ColumnTransformer ---
    # Transformers numéricos y categóricos
    transformers = []
    transformers.extend(numerical_transformers)     
    transformers.extend(categorical_transformers)

    # Construimos el ColumnTransformer final
    # que une pipelines numéricos + pipelines categóricos
    col_trans_preprocessor = ColumnTransformer(
        transformers=transformers, # lista de tuplas (name, pipeline, cols)
        remainder="drop", # elimina columnas no especificadas
        verbose_feature_names_out=True) # nombres detallados de columnas
    return col_trans_preprocessor


#### Construir pipelines

In [27]:
# Creación y evaluación del pipeline
# Construcción del pipeline con preprocesador y modelo

def make_pipeline(preprocessor: ColumnTransformer, model=None):
    """Construye un Pipeline con el preprocesador y el modelo indicado.
    Args:
        preprocessor (ColumnTransformer): Preprocesador ColumnTransformer
        model (_type_, optional): Modelo de clasificación. Defaults to None.
    Returns:
        Pipeline: Pipeline con preprocesador y modelo, si no se indica modelo
        se usa LinearDiscriminantAnalysis por defecto.
    """
    if model is None:
        model = LinearDiscriminantAnalysis()
        #model = LogisticRegression(max_iter=10000, random_state=RANDOM_STATE)
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model),
    ])




#### Evaluar pipelines

Esta función evalua el pipeline usando validación cruzada estratificada

In [28]:
def evaluate_pipeline(pipe: Pipeline, X: pd.DataFrame, y: pd.Series, n_splits=5):
    """ Evalúa el pipeline usando Validación Cruzada estratificada
    Args:
        pipe (Pipeline): Pipeline a evaluar.
        X (pd.DataFrame): Datos de entrada.
        y (pd.Series): Datos objetivo.
        n_splits (int, optional): número de divisiones para Stratified K-Fold. Por defecto es 5.
    Returns:
        dict: Diccionario con las métricas promedio y desviación estándar.
    """
    # Configuramos la Validación Local Cruzada  n_splits splits (divisiones)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    # Definimos las métricas que queremos extraer
    # f1, roc_auc, precision, recall, accuracy son strings estándar de sklearn.
    # Kappa requiere make_scorer.
    scoring_metrics = {
        "f1": "f1",
        "roc_auc": "roc_auc",
        "accuracy": "accuracy",
        'kappa': make_scorer(cohen_kappa_score),
        'precision': 'precision',
        'recall': 'recall',
    }
    cv_results = cross_validate(pipe, X, y, cv=cv, scoring=scoring_metrics, n_jobs=-1)
    return {
        "f1_mean": cv_results["test_f1"].mean(),
        "f1_std":  cv_results["test_f1"].std(),
        "auc_mean": cv_results["test_roc_auc"].mean(),
        "auc_std":  cv_results["test_roc_auc"].std(),
        "accuracy_mean": cv_results["test_accuracy"].mean(),
        "accuracy_std":  cv_results["test_accuracy"].std(),
        "kappa_mean": cv_results["test_kappa"].mean(),
        "kappa_std":  cv_results["test_kappa"].std(),
        "precision_mean": cv_results["test_precision"].mean(),
        "precision_std":  cv_results["test_precision"].std(),
        "recall_mean": cv_results["test_recall"].mean(),
        "recall_std":  cv_results["test_recall"].std()
    }

#### Evaluar varios modelos

In [29]:

def benchmark_models_with_fixed_preprocess(X: pd.DataFrame, y: pd.Series, models: dict, 
                                           best_preprocesor_version: str, n_splits=5):
    """Evalúa varios modelos con un preprocesador fijo usando Validación Cruzada.
    Args:
        X (pd.DataFrame): Datos de entrada.
        y (pd.Series): Datos objetivo.
        models (dict): Diccionario con nombre y objeto del modelo a evaluar.
        best_preprocesor_version (str): Versión del preprocesador a usar.
        n_splits (int, optional): número de divisiones para Stratified K-Fold. Por defecto es 5.
    Returns:
        pd.DataFrame: DataFrame con resultados de cada modelo evaluado.
    """
    # Construimos el preprocesador fijo con la mejor versión
    best_preprocesor = make_preprocessor(X, best_preprocesor_version,NUM_FEATURES,CAT_FEATURES)

    model_results = []  # Lista para almacenar resultados de cada modelo
    # Evaluamos cada modelo con el preprocesador fijo
    for name, model in models.items():
        # Construimos el pipeline con preprocesador fijo y el modelo actual
        pipe = Pipeline([("preprocessor", best_preprocesor), ("classifier", model)])
        try:
            # Evaluamos el pipeline con validación cruzada para el pipeline actual
            cv_metrics = evaluate_pipeline(pipe, X, y , n_splits=n_splits)
            model_results.append({
                "model": name,
                "preprocessor_version": best_preprocesor_version,
                **cv_metrics
            })
        except Exception as e:
            model_results.append({"model": name, "error": str(e)})
    # Construimos el DataFrame de resultados ordenado por F1 medio
    out = pd.DataFrame(model_results).sort_values(by="f1_mean", ascending=False, na_position="last")
    return out


------------------
## Evaluación de diferentes preprocesadores y modelos

A partir de aquí comenzamos la evaluación de los distintos preprocesadores que se han configurado en la función `make_preprocesor` y modelos, configurador `benchmark_models_with_fixed_preprocess`

In [ ]:
# Experimentos: combinaciones de preprocesamiento a probar

EXPERIMENTS = [
    #"N1_C0",  # num only (simple imputer mediana)
    #"N2_C0",  # num only (simple imputer mediana) + indicator
    #"N3_C0",  # num only (simple imputer mediana) + indicator + scaler
    "N3_C1",  # num only (simple imputer mediana + indicator + scaler) + cat (onehot) sin surname
    "N4_C1",  # num (robust scaler) + cat (onehot) sin surname
    "N5_C1",  # num (KNN imputer 5 vecinos) + cat (onehot) sin surname
    "N6_C1",  # num (KNN imputer + robust scaler) + cat (onehot) sin surname
    "N7_C1",  # num (KNN imputer + robust scaler) + cat (onehot) sin surname
    "N8_C1",  # num (KNN imputer 9 vecinos) + cat (onehot) sin surname
    # No mejora -> Volvemos a la base N5_C1
    "N9_C1",  # num (KNN imputer 5 vecinos + indicador) + cat (onehot) sin surname
    "N10_C1",  # num (KNN imputer 5 vecinos + indicador + pesos distancia) + cat (onehot) sin surname
]

# Modelos a probar
models = {
    # Modelos lineales
    "LinearDiscriminantAnalysis": LinearDiscriminantAnalysis(),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"),
    # Arboles
    # No necesitan escalado de variables
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=400,random_state=RANDOM_STATE, 
                                           class_weight="balanced",n_jobs=-1,),
    "RandomForest_bal_subsample": RandomForestClassifier(n_estimators=400,random_state=RANDOM_STATE, 
                                           class_weight="balanced_subsample",n_jobs=-1,),
    "NaiveBayes": GaussianNB(),
    "RedesNeurales": MLPClassifier(hidden_layer_sizes=(50,30), max_iter=1000, random_state=RANDOM_STATE),
    "KNN_5": KNeighborsClassifier(n_neighbors=5, n_jobs=-1,),
    "KNN_5_distance": KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1),

    
}


#### Búsqueda del mejor pipeline

Evaluamos las diferentes configuraciones de preprocesador que tenemos configuradas con el modelo por defecto definido en la función `make_pipeline` con el objetivo de obtener mejor versión o combinación de preprocesado. 

In [31]:
preprocesors_results = []
for experiment_version in EXPERIMENTS:
    best_preprocesor = make_preprocessor(train_df, experiment_version,NUM_FEATURES,CAT_FEATURES)
    pipe = make_pipeline(best_preprocesor) # Construimos el pipeline con el modelo por defecto
    cv_metrics = evaluate_pipeline(pipe, X_train, y_train , n_splits=5)
    preprocesors_results.append({"version": experiment_version, **cv_metrics})
    #print(experiment_version, metrics)

results_df = pd.DataFrame(preprocesors_results).sort_values("f1_mean", ascending=False)
display(results_df)
best_preprocesor_version = results_df.iloc[0]["version"]
print("Mejor versión de preprocesador:", best_preprocesor_version)

,version,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
2,N5_C1,0.332133,0.016951,0.770777,0.008065,0.808625,0.004832,0.243088,0.017389,0.576259,0.032785,0.233742,0.015325
3,N6_C1,0.332133,0.016951,0.770777,0.008065,0.808625,0.004832,0.243088,0.017389,0.576259,0.032785,0.233742,0.015325
6,N9_C1,0.331837,0.015740,0.770065,0.009218,0.808375,0.004268,0.242534,0.015610,0.574326,0.029009,0.233742,0.014570
7,N10_C1,0.331705,0.014193,0.769637,0.009104,0.808250,0.003820,0.242265,0.013792,0.573353,0.026525,0.233742,0.013636
4,N7_C1,0.329124,0.015500,0.770609,0.008378,0.808000,0.004750,0.239975,0.016288,0.572756,0.031957,0.231288,0.013800
0,N3_C1,0.328859,0.016720,0.768109,0.008109,0.807250,0.005654,0.238822,0.018148,0.568047,0.037219,0.231902,0.014596
5,N8_C1,0.328164,0.014743,0.770968,0.008158,0.807625,0.005426,0.238768,0.016483,0.571020,0.036447,0.230675,0.012781
1,N4_C1,0.325738,0.016948,0.768633,0.008256,0.807125,0.005297,0.236286,0.017750,0.568027,0.035417,0.228834,0.015227


Mejor versión de preprocesador: N5_C1


#### Búsqueda del mejor modelo

Evaluamos los modelos con el mejor preprocesador encontrado 

In [32]:
# Evaluamos los modelos con el mejor preprocesador encontrado
models_df = benchmark_models_with_fixed_preprocess(X_train, y_train, models, best_preprocesor_version, n_splits=5)
print("---- Resultados de validación cruzada de modelos con preprocesador fijo:----")
display(models_df)

best_model_name = models_df.iloc[0]["model"]
print("Mejor versión de preprocesador:", best_preprocesor_version)
print("Mejor modelo:", best_model_name)

---- Resultados de validación cruzada de modelos con preprocesador fijo:----


,model,preprocessor_version,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
3,RandomForest,N5_C1,0.522415,0.016983,0.838010,0.006107,0.850500,0.002417,0.443251,0.016380,0.747942,0.005968,0.401840,0.020805
4,RandomForest_bal_subsample,N5_C1,0.522402,0.021763,0.837895,0.006427,0.850625,0.003536,0.443427,0.021188,0.749385,0.016295,0.401840,0.027368
5,RedesNeurales,N5_C1,0.516047,0.017945,0.796620,0.014852,0.812000,0.003881,0.399793,0.019240,0.542419,0.009572,0.492638,0.027340
1,LogisticRegression,N5_C1,0.496705,0.011084,0.771742,0.008602,0.713625,0.005440,0.318451,0.013778,0.386904,0.007245,0.693865,0.024494
6,KNN_5,N5_C1,0.466473,0.018760,0.769091,0.004423,0.828250,0.005235,0.372665,0.020689,0.635524,0.023352,0.368712,0.018137
7,KNN_5_distance,N5_C1,0.465587,0.020538,0.771627,0.004428,0.826250,0.005742,0.369759,0.022439,0.623881,0.025209,0.371779,0.021021
2,DecisionTree,N5_C1,0.459049,0.026751,0.659833,0.016454,0.783250,0.010705,0.323619,0.032902,0.467151,0.027064,0.451534,0.028736
0,LinearDiscriminantAnalysis,N5_C1,0.332133,0.016951,0.770777,0.008065,0.808625,0.004832,0.243088,0.017389,0.576259,0.032785,0.233742,0.015325


Mejor versión de preprocesador: N5_C1
Mejor modelo: RandomForest


## Construcción del pipeline final para Kaggle

Una vez obtenido la mejor combinación de preprocesadores y el mejor modelo, construimos el pipeline final para kaggle con la mejor combinación de ambos y volvemos a ejecutar la evaluación y el entrenamiento para finalmente obtener la predicción y generar el fichero para kaggle. 

In [33]:
# Construcción del pipeline final para Kaggle con el mejor preprocesador y modelo
# Construimos el preprocesador fijo con la mejor versión
best_preprocesor = make_preprocessor(X_train, best_preprocesor_version,NUM_FEATURES,CAT_FEATURES)
best_model = models[best_model_name]
# Pipeline Completo (Preprocesamiento + Modelo)
best_model_pipeline = Pipeline(steps=[
    ('preprocessor', best_preprocesor),
    ('classifier', best_model)
])
# Configuramos y ejecutamos la Validación Cruzada local
cv_metrics = evaluate_pipeline(best_model_pipeline, X_train, y_train, n_splits=5)
# Generación de Submission para Kaggle con el mejor modelo encontrado
# Re-entrenamos con TODOS los datos de train para la predicción final
best_model_pipeline.fit(X_train, y_train) 
test_predictions = best_model_pipeline.predict(test_df)

# Crear fichero de salida
submission_df = pd.DataFrame({
    'CustomerId': test_df['CustomerId'],
    'Exited': test_predictions
})
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Fichero '{SUBMISSION_PATH}' generado correctamente.")

print("\n---- Mejores Resultados y Validación Cruzada local -----")
print("Mejor modelo:", best_model_name)
print("Mejor versión de preprocesador:", best_preprocesor_version)
print(f"Mean F1-Score:  {cv_metrics['f1_mean']:.4f} (+/- Std {cv_metrics['f1_std']:.4f})")
print(f"Mean Accuracy:  {cv_metrics['accuracy_mean']:.4f} (+/- Std {cv_metrics['accuracy_std']:.4f})")
print(f"Mean Kappa:     {cv_metrics['kappa_mean']:.4f}")
print(f"Mean Precision: {cv_metrics['precision_mean']:.4f}")
print(f"Mean Recall:    {cv_metrics['recall_mean']:.4f}")



Fichero '/kaggle/working/submission.csv' generado correctamente.

---- Mejores Resultados y Validación Cruzada local -----
Mejor modelo: RandomForest
Mejor versión de preprocesador: N5_C1
Mean F1-Score:  0.5224 (+/- Std 0.0170)
Mean Accuracy:  0.8505 (+/- Std 0.0024)
Mean Kappa:     0.4433
Mean Precision: 0.7479
Mean Recall:    0.4018
